# FPP01 — Evaluation Protocol: Subject-Disjoint Splits, Verification Pairs and the Shared Library (3 repetitions)

**First kernel of the FPP paper series** (FingerPrint-Final workspace; CASIA not used). It fixes, verifies and documents everything every later kernel relies on:

1. **Installs `fpp_lib.py`** into `output/00_code/` (protocol constants, source-balanced pair sampler, paper metrics, frozen-threshold evaluation, standard figures). Later kernels load it from there.
2. **Pair files for the 3 repetitions** in `FPP01_…/protocol/<rep>/pairs/` (one folder read by every kernel):
   * from FPF00, unchanged: `DB1_train`, `DB1_validation`, `DB1_test`, `DB2_test`, `DB1xDB2_test`, `FingerDOS_test`;
   * joint DB1 + DB2 files: `DB2_train`, `DB1xDB2_train`, `DB2_validation`, `DB1xDB2_validation`, regenerated with the FPF02 sampler and seeds and **checked to be identical** to the FPF02 pilot files when those exist.
   Every file is hashed (sha256) in a manifest; kernels refuse a changed file.
3. **Checks:** train / validation / adaptation / test subjects disjoint; every image of every pair set inside its partition; cross-sensor genuine pairs are the same finger; no duplicate or self pairs; impostor mix (25 % same subject, 75 % other subjects); FingerDOS = all 60 subjects; the 5 calibration hold-out subjects ('adaptation' in the split files) unused by every pair set.
4. **Documents:** splits per repetition, images and pairs per set, how often each subject is in each partition, figures.
5. **Materialized training augmentation:** **3 augmented copies of every MOLF DB1 and DB2 image**, written to *new* files `FPP01_…/augmentation/<dataset>_augmented_x3_224.h5` — the original FPF00 image caches are never modified (their sha256 is checked before and after). Recipe = the FPM04C / FPF02 **mild** augmentation (rotation ±5°, shift ±3 %, brightness ±10 %, contrast ±15 %, Gaussian noise 6 grey levels, white fill, no zoom); every copy is augmented, each from its own fixed seed `[20261107, dataset, image, copy]`, so the copies are reproducible and identical for every kernel. The parameters of every copy are stored. Training kernels (FPP04 onward) draw, for each image of a training pair, the original or one of its 3 copies; validation, test, FingerDOS and the training scores for the threshold always use the originals. FingerDOS is never augmented.

**Protocol fixed for the whole series:** 60 train / 15 validation / 5 calibration hold-out (bias check of the threshold only; 'adaptation' in the split files) / 20 test MOLF subjects, the same partition for DB1 and DB2, 3 reproducible repetitions; training on DB1, DB2 and DB1×DB2 pairs; early stopping, LR drops, best epoch and selection on the **source-balanced validation EER** (DB1, DB2, DB1×DB2 validation weighted equally, one threshold; AUC breaks ties), validation only; **FingerDOS = external test**; one frozen threshold = **source-balanced EER of the validation pairs** (DB1, DB2, DB1×DB2 weighted equally), raw scores, no centering, applied unchanged to MOLF test and FingerDOS (validation rows = development numbers). **Paper metrics: AUC, EER, FMR, FNMR, F1-macro, accuracy, precision, recall** + time. **Standard figures:** ROC (linear and log FMR), DET, FMR / FNMR vs threshold (EER crossing), confusion matrix, score distributions, metric bars.

**Needs:** FPF00 finished (splits, FPF00 pair files, image caches, templates). **Hardware:** CPU, High-RAM. ≈ 15–25 min (most of it writing the 24,000 augmented images; resumable).

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install `fpp_lib.py` into `output/00_code/`

In [ ]:
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
import base64, hashlib
FPP_LIB_SHA256 = '76e5c8385bb727bcbb1671f1cfdfcba8f6bcd7471051e4bf05788983163c22a4'
FPP_LIB_B64 = '...'
_src = base64.b64decode(FPP_LIB_B64)
if hashlib.sha256(_src).hexdigest() != FPP_LIB_SHA256: raise RuntimeError('embedded fpp_lib corrupted')
dst = CODE / 'fpp_lib.py'
if dst.is_file() and hashlib.sha256(dst.read_bytes()).hexdigest() != FPP_LIB_SHA256:
    bak = CODE / f'fpp_lib.previous_{hashlib.sha256(dst.read_bytes()).hexdigest()[:10]}.py'; shutil.copy2(dst, bak); print('previous fpp_lib kept as', bak.name)
dst.write_bytes(_src); print('Installed', dst, 'sha256', FPP_LIB_SHA256[:16])
for m in ('fpf_lib.py', 'fpp_lib.py'): shutil.copy2(CODE / m, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
ENV = L.env_info(); print('fpp_lib', P.VERSION)

## 2. Configuration

In [ ]:
import matplotlib.pyplot as plt
KERNEL = 'FPP01'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'; FIG.mkdir(exist_ok=True)
PDIR = P.protocol_dir(ws); LOG = L.RunLog(KDIR / 'FPP01_run_log.csv'); REPS = L.repetitions(ws)
F02_NAME = 'FPF02_MobileNetV3Large_Siamese_FullFT_MildAug_DB1DB2JointTrain_CrossSensorPairs_FMR1pctThreshold_FingerDOSTest'
F02DIR = next((d for d in (ws['output'] / F02_NAME, ws['output'] / 'archive' / F02_NAME) if d.is_dir()), ws['output'] / F02_NAME)   # FPF02 pilot (also found in output/archive/)
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], protocol=P.PROTOCOL_TABLE, fpp_lib_version=P.VERSION, fpp_lib_sha256=FPP_LIB_SHA256, repetitions=REPS, pair_sets=P.PAIR_SETS,
              joint_pair_files=P.JOINT_PAIR_FILES, train_sources=P.TRAIN_SOURCES, validation_sources=P.VAL_SOURCES, test_sets=P.TEST_SETS, groups=P.GROUPS,
              threshold=P.THRESHOLD_RULE, metrics=P.METRICS, selection_set=P.SELECTION_SET)
print(json.dumps(CONFIG, indent=2, default=str))

## 3. Write the 10 pair sets per repetition (FPF00 files + regenerated joint files, compared with FPF02)

In [ ]:
def same_arrays(a, b): return all(np.array_equal(np.asarray(a[k]), np.asarray(b[k])) for k in ('index_a', 'index_b', 'label', 'negative_type'))
MAN = {}
for rep in REPS:
    d = PDIR / rep; mf = d / 'fpp_pairs_manifest.json'
    if mf.is_file(): MAN[rep] = json.loads(mf.read_text()); print(rep, 'manifest exists -> kept'); continue
    proto = L.load_protocol(ws, rep); (d / 'pairs').mkdir(parents=True, exist_ok=True); t = L.timer()
    pairs = {n: proto['pairs'][n] for n in P.PAIR_SETS if n not in P.JOINT_PAIR_FILES}; joint = P.build_joint_pairs(proto); pairs.update(joint)
    match = {}
    for n in P.JOINT_PAIR_FILES:
        f = F02DIR / 'protocol' / rep / 'pairs' / f'{n}_pairs.npz'; match[n] = (same_arrays(joint[n], L.load_npz(f)) if f.is_file() else None)
        if match[n] is False: raise RuntimeError(f'{rep} {n}: regenerated pairs differ from the FPF02 pilot file')
    files = {}
    for n in P.PAIR_SETS:
        f = d / 'pairs' / f'{n}_pairs.npz'; np.savez_compressed(f, **{k: np.asarray(v) for k, v in pairs[n].items()})
        files[n] = {'file': L.rel(ws, f), 'sha256': L.sha256_of(f), 'source': 'FPF00' if n not in P.JOINT_PAIR_FILES else 'FPP01 (FPF02 sampler and seeds)',
                    'identical_to_FPF02_pilot': match.get(n), 'pairs': int(len(pairs[n]['label'])), 'genuine': int(np.sum(pairs[n]['label']))}
    MAN[rep] = {'repetition': rep, 'seed': proto['seed'], 'fpp_lib_sha256': FPP_LIB_SHA256, 'files': files, 'rng_streams': {n: [proto['seed'], 1000 + k] for k, n in enumerate(P.JOINT_PAIR_FILES)},
                'written_utc': pd.Timestamp.utcnow().isoformat()}
    mf.write_text(json.dumps(MAN[rep], indent=2)); LOG.add(step='pairs', repetition=rep, status='ok', seconds=round(t(), 1), identical_to_FPF02=json.dumps(match))
    print(rep, 'written | identical to FPF02 pilot:', match)

## 4. Checks (leakage, partitions, cross-sensor identity, duplicates, impostor mix)

In [ ]:
CHECKS = []
def check(rep, name, ok, detail=''):
    CHECKS.append({'repetition': rep, 'check': name, 'passed': bool(ok), 'detail': detail})
    if not ok: print('FAILED', rep, name, detail)
for rep in REPS:
    proto, Pp = P.load_pairs(ws, rep)                                  # also verifies every sha256
    sp = proto['splits'].assign(subject_id=proto['splits'].subject_id.astype(str)); part = sp.set_index('subject_id').partition
    groups = {p: set(sp.subject_id[sp.partition == p]) for p in sp.partition.unique()}
    check(rep, 'MOLF partitions are disjoint', all(not (groups[a] & groups[b]) for a in groups for b in groups if a < b), json.dumps({k: len(v) for k, v in groups.items()}))
    check(rep, 'partition sizes 60 / 15 / 5 / 20', sorted(len(v) for v in groups.values()) == [5, 15, 20, 60], json.dumps({k: len(v) for k, v in groups.items()}))
    S = {d: proto['samples'][d] for d in ('MOLF_DB1', 'MOLF_DB2', 'FingerDOS')}
    for d in ('MOLF_DB1', 'MOLF_DB2'): check(rep, f'{d}: sample_index = row', np.array_equal(S[d].sample_index.to_numpy(), np.arange(len(S[d]))))
    used = set()
    for n, (da, db, prt) in P.PAIR_SETS.items():
        p = Pp[n]; a, b, y = np.asarray(p['index_a']), np.asarray(p['index_b']), np.asarray(p['label'])
        if prt != 'external_test':
            sa = set(S[da].subject_id.astype(str).iloc[np.unique(a)]); sb = set(S[db].subject_id.astype(str).iloc[np.unique(b)]); used |= sa | sb
            check(rep, f'{n}: every image in the {prt} partition', {part[s] for s in sa | sb} == {prt})
        else: check(rep, f'{n}: all 60 FingerDOS subjects', S['FingerDOS'].subject_id.astype(str).iloc[np.unique(np.r_[a, b])].nunique() == 60)
        ida = S[da].identity_id.astype(str).to_numpy()[a]; idb = S[db].identity_id.astype(str).to_numpy()[b]
        check(rep, f'{n}: genuine = same finger, impostor = different finger', (ida[y == 1] == idb[y == 1]).all() and (ida[y == 0] != idb[y == 0]).all())
        keys = set(zip(a.tolist(), b.tolist())) if da != db else set(zip(np.minimum(a, b).tolist(), np.maximum(a, b).tolist()))
        check(rep, f'{n}: no duplicate pairs', len(keys) == len(y)); check(rep, f'{n}: no self pairs', da != db or not (a == b).any())
        nt = np.asarray(p['negative_type'])[y == 0]; frac = float((nt == 1).mean()) if len(nt) else np.nan
        check(rep, f'{n}: same-subject impostor fraction ≈ 0.25', abs(frac - 0.25) < 0.02, f'{frac:.4f}')
    check(rep, 'calibration hold-out subjects unused by every pair set', not (used & groups.get('adaptation', set())))
checks = pd.DataFrame(CHECKS); print(f'{int(checks.passed.sum())} / {len(checks)} checks passed')
if not checks.passed.all(): raise RuntimeError('protocol checks failed: see the table above')

## 5. Documentation tables and figures

In [ ]:
SPL, PAIRS, IMGS = [], [], []
for rep in REPS:
    proto, Pp = P.load_pairs(ws, rep, verify=False); sp = proto['splits']
    for prt, g in sp.groupby('partition'): SPL.append({'repetition': rep, 'partition': prt, 'subjects': len(g), 'subject_ids': ', '.join(map(str, sorted(g.subject_id.astype(str))))})
    for d in ('MOLF_DB1', 'MOLF_DB2', 'FingerDOS'):
        s = proto['samples'][d]
        for prt, g in s.groupby('partition'): IMGS.append({'repetition': rep, 'dataset': d, 'partition': prt, 'images': len(g), 'fingers': g.identity_id.nunique(), 'subjects': g.subject_id.nunique()})
    for n, p in Pp.items():
        y = np.asarray(p['label']); nt = np.asarray(p['negative_type'])
        PAIRS.append({'repetition': rep, 'pair_set': n, 'dataset_a': P.PAIR_SETS[n][0], 'dataset_b': P.PAIR_SETS[n][1], 'partition': P.PAIR_SETS[n][2], 'role': P.ROLE[n],
                      'pairs': len(y), 'genuine': int(y.sum()), 'impostor_same_subject': int((nt == 1).sum()), 'impostor_other_subject': int((nt == 2).sum()),
                      'source': MAN[rep]['files'][n]['source'], 'sha256': MAN[rep]['files'][n]['sha256'][:16]})
splits = pd.DataFrame(SPL); pair_table = pd.DataFrame(PAIRS); image_table = pd.DataFrame(IMGS)
display(pair_table[pair_table.repetition == REPS[0]].drop(columns=['repetition']))
# subject x repetition partition map
sp0 = pd.concat([L.load_protocol(ws, r)['splits'].assign(repetition=r) for r in REPS]); sp0['subject_id'] = sp0.subject_id.astype(str)
pm = sp0.pivot(index='subject_id', columns='repetition', values='partition'); pm = pm.loc[sorted(pm.index, key=lambda s: (len(s), s))]
codes = {'train': 0, 'validation': 1, 'adaptation': 2, 'test': 3}; M = pm.replace(codes).to_numpy(float)
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(4 + len(REPS), 14)); cm = ListedColormap([P.COLORS[0], P.COLORS[1], P.COLORS[7], P.COLORS[3]]); ax.imshow(M, aspect='auto', cmap=cm, vmin=-.5, vmax=3.5, interpolation='nearest')
ax.set_xticks(range(len(REPS))); ax.set_xticklabels([r[:6] for r in REPS]); ax.set_yticks(range(len(pm))); ax.set_yticklabels(pm.index, fontsize=5); ax.set_ylabel('MOLF subject')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=cm(v), label=k) for k, v in codes.items()], fontsize=7, loc='upper right', bbox_to_anchor=(1.6, 1)); ax.set_title('Subject partitions per repetition (same for DB1 and DB2)', fontsize=9)
P._save(fig, FIG / 'subject_partitions.png')
freq = sp0.groupby(['subject_id', 'partition']).size().unstack(fill_value=0); subject_frequency = freq.reset_index()
q = pair_table[pair_table.repetition == REPS[0]].set_index('pair_set').reindex(list(P.PAIR_SETS)); fig, ax = plt.subplots(figsize=(11, 4)); x = np.arange(len(q))
ax.bar(x, q.genuine, color=P.COLORS[0], label='genuine'); ax.bar(x, q.impostor_same_subject, bottom=q.genuine, color=P.COLORS[1], label='impostor, same subject')
ax.bar(x, q.impostor_other_subject, bottom=q.genuine + q.impostor_same_subject, color=P.COLORS[3], label='impostor, other subject')
ax.set_xticks(x); ax.set_xticklabels(q.index, rotation=30, ha='right', fontsize=8); ax.set(ylabel='pairs', title=f'Pairs per set ({REPS[0][:6]}; identical structure in every repetition)'); ax.legend(fontsize=8); ax.grid(alpha=.3, axis='y')
P._save(fig, FIG / 'pairs_per_set.png')
fig, ax = plt.subplots(figsize=(8, 3.8)); it = image_table[image_table.repetition == REPS[0]].pivot(index='dataset', columns='partition', values='images').fillna(0)
it.plot.bar(ax=ax, color=[P.COLORS[i] for i in range(len(it.columns))]); ax.set(ylabel='images', title=f'Images per dataset and partition ({REPS[0][:6]})'); ax.grid(alpha=.3, axis='y'); ax.tick_params(axis='x', rotation=0)
P._save(fig, FIG / 'images_per_partition.png')

## 6. Materialized training augmentation: 3 extra copies per MOLF image (new files; originals untouched)

In [ ]:
import h5py
AUG_ROWS, AUG_BLOCK = [], 250
for d in P.AUG_DATASETS:
    src_path = ws['images'] / f'{d}_standardized_grayscale_224.h5'; sha_before = L.sha256_of(src_path)
    with h5py.File(L.local_copy(src_path), 'r') as h:
        if not h['complete'][:].all(): raise RuntimeError(f'{d}: FPF00 image cache incomplete')
        X = h['images'][:]
    n = len(X); out = P.augmentation_file(ws, d); out.parent.mkdir(parents=True, exist_ok=True); t = L.timer()
    with h5py.File(out, 'a') as g:
        if 'images' not in g:
            g.create_dataset('images', (n, P.AUG_COPIES) + X.shape[1:], np.uint8, chunks=(1, 1) + X.shape[1:], compression='gzip', compression_opts=4)
            g.create_dataset('params', (n, P.AUG_COPIES, len(P.AUG_PARAM_NAMES)), np.float32); g.create_dataset('complete', (n,), bool)
            g.attrs['recipe'] = json.dumps(P.AUG_RECIPE); g.attrs['seed'] = P.AUG_SEED; g.attrs['copies'] = P.AUG_COPIES; g.attrs['param_names'] = json.dumps(P.AUG_PARAM_NAMES)
            g.attrs['source'] = L.rel(ws, src_path); g.attrs['source_sha256'] = sha_before; g.attrs['dataset'] = d
        if g.attrs['source_sha256'] != sha_before: raise RuntimeError(f'{d}: FPF00 image cache changed since the augmentation was started')
        done = g['complete'][:]
        for s in range(0, n, AUG_BLOCK):
            if done[s:s + AUG_BLOCK].all(): continue
            A, Q = zip(*[P.augment_copies(X[i], d, i) for i in range(s, min(n, s + AUG_BLOCK))])
            g['images'][s:s + len(A)] = np.stack(A); g['params'][s:s + len(A)] = np.stack(Q); g['complete'][s:s + len(A)] = True
            print(f'{d}: {min(n, s + AUG_BLOCK)} / {n}', end='\r')
        Aall = g['images']; i0 = [0, n // 2, n - 1]
        recheck = all(np.array_equal(Aall[i], P.augment_copies(X[i], d, i)[0]) for i in i0)        # reproducible from the seed
        diff = np.array([np.abs(Aall[i].astype(np.int16) - X[i][None].astype(np.int16)).mean() for i in range(0, n, max(1, n // 200))])
        prm = g['params'][:]
    sha_after = L.sha256_of(src_path)
    if sha_after != sha_before: raise RuntimeError(f'{d}: the original image cache changed!')
    AUG_ROWS.append({'dataset': d, 'images': n, 'copies_per_image': P.AUG_COPIES, 'augmented_images': n * P.AUG_COPIES, 'file': L.rel(ws, out), 'file_mb': round(out.stat().st_size / 2**20, 1),
                     'originals_unchanged': sha_after == sha_before, 'reproducible_from_seed': recheck, 'mean_abs_grey_difference_vs_original': round(float(diff.mean()), 2),
                     **{f'{p}_min': round(float(prm[..., k].min()), 3) for k, p in enumerate(P.AUG_PARAM_NAMES)}, **{f'{p}_max': round(float(prm[..., k].max()), 3) for k, p in enumerate(P.AUG_PARAM_NAMES)},
                     'seconds': round(t(), 1)})
    LOG.add(step='augmentation', dataset=d, status='ok', images=n * P.AUG_COPIES, seconds=round(t(), 1))
    # figure: originals and their 3 copies
    rng = np.random.default_rng(0); pick = rng.choice(n, 4, replace=False)
    with h5py.File(out, 'r') as g: A4 = g['images'][sorted(pick)]
    fig, axs = plt.subplots(4, 1 + P.AUG_COPIES, figsize=(2.6 * (1 + P.AUG_COPIES), 10.5))
    for r_, i in enumerate(sorted(pick)):
        axs[r_, 0].imshow(X[i][..., 0], cmap='gray', vmin=0, vmax=255); axs[r_, 0].set_title(f'original #{i}', fontsize=8)
        for k in range(P.AUG_COPIES): axs[r_, k + 1].imshow(A4[r_, k][..., 0], cmap='gray', vmin=0, vmax=255); axs[r_, k + 1].set_title(f'copy {k + 1}', fontsize=8)
    for a in axs.ravel(): a.axis('off')
    fig.suptitle(f'{d}: original + {P.AUG_COPIES} materialized augmented copies', fontsize=10); P._save(fig, FIG / f'augmentation_examples_{d}.png')
    fig, axs = plt.subplots(1, len(P.AUG_PARAM_NAMES), figsize=(3.2 * len(P.AUG_PARAM_NAMES), 2.8))
    for k, (a, p) in enumerate(zip(axs, P.AUG_PARAM_NAMES)): a.hist(prm[..., k].ravel(), 40, color=P.COLORS[k % 10]); a.set_title(p, fontsize=8); a.grid(alpha=.3)
    fig.suptitle(f'{d}: augmentation parameters of all {n * P.AUG_COPIES} copies', fontsize=10); P._save(fig, FIG / f'augmentation_parameters_{d}.png')
augmentation_table = pd.DataFrame(AUG_ROWS); display(augmentation_table)
if not (augmentation_table.originals_unchanged.all() and augmentation_table.reproducible_from_seed.all()): raise RuntimeError('augmentation checks failed')

## 7. Self-test of the paper metrics and the standard figures (synthetic scores)

In [ ]:
rng = np.random.default_rng(0); demo = {}
for n in P.PAIR_SETS:
    y = np.r_[np.ones(400, int), np.zeros(400, int)]; s = np.r_[rng.normal(.7, .1, 400), rng.normal(.4, .12, 400)]; demo[n] = (y, s)
rows, thr = P.evaluate(demo, 'self_test', 'demo', 'rep_demo'); m = pd.DataFrame(rows).set_index('set').loc['DB1_test']
y, s = demo['DB1_test']; pred = s >= thr; acc = (pred == y).mean(); prec = (pred & (y == 1)).sum() / pred.sum(); rec = (pred & (y == 1)).sum() / (y == 1).sum()
assert abs(m.accuracy - acc) < 1e-12 and abs(m.precision - prec) < 1e-12 and abs(m.recall - rec) < 1e-12 and abs(m.recall - (1 - m.fnmr)) < 1e-12
rd = KDIR / '_self_test' / 'runs' / 'rep_demo' / 'demo'; rd.mkdir(parents=True, exist_ok=True); P.save_run(rd, demo, rows, thr, {'note': 'synthetic self-test'})
P.standard_figures(KDIR / '_self_test' / 'figures', {'synthetic demo': [rd]}, sets=['DB1_test'])
print('metrics and figures OK | metric columns:', P.METRICS)

## 8. Workbook and protocol manifest

In [ ]:
(KDIR / 'protocol_manifest.json').write_text(json.dumps({'kernel': KERNEL, 'fpp_lib_sha256': FPP_LIB_SHA256, 'repetitions': {r: MAN[r] for r in REPS}, 'checks_passed': bool(checks.passed.all()),
                                                       'n_checks': int(len(checks)), 'threshold': P.THRESHOLD_RULE, 'metrics': P.METRICS,
                                                       'augmentation': {'recipe': P.AUG_RECIPE, 'copies': P.AUG_COPIES, 'seed': P.AUG_SEED, 'files': augmentation_table.to_dict('records')}}, indent=2, default=str))
L.write_workbook(KDIR / 'FPP01_Protocol.xlsx', {'Pair Sets': pair_table, 'Splits': splits, 'Images': image_table, 'Subject Frequency': subject_frequency, 'Checks': checks, 'Augmentation': augmentation_table, 'Augmentation Recipe': P.AUG_RECIPE,
                                               'Configuration': CONFIG, 'Run Log': LOG.frame(), 'Environment': ENV, 'Figure Index': L.figure_index(KDIR)})
pair_table.to_csv(KDIR / 'FPP01_pair_sets.csv', index=False); checks.to_csv(KDIR / 'FPP01_checks.csv', index=False)
print('FPP01 complete:', len(REPS), 'repetitions,', int(checks.passed.sum()), 'checks passed')